# Auto-Label Notebook — Prompt Engineering & Face Cleaning
**AMD OnClick AI / ROCm 6.4 compatible**

Test prompt engineering for text generation and face clustering for album cleaning.

| # | Section | What you do |
|---|---------|-------------|
| 0 | Setup | Install packages |
| 1 | Load Data | Load 100 images from DownFlow/meizi |
| 2 | Prompt Eng | Interactive prompt engineering with Qwen |
| 3 | Face Detect | Test DeepFace face detection |
| 4 | Cluster | Test DBSCAN clustering |
| 5 | Pipeline | Full cleaning pipeline demo |

---
## 0 · Setup — Install Dependencies

> Run once per environment.

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

# Torch — AMD ROCm 6.4
pip('torch', 'torchvision', '--index-url', 'https://download.pytorch.org/whl/rocm6.4', '--upgrade')

# Core
pip('transformers>=5.0.0', 'huggingface_hub>=1.9.0',
    'datasets>=3.0.0', 'pandas>=2.0.0', 'Pillow>=10.0.0',
    'numpy>=1.24.0', 'scikit-learn>=1.4.0')
pip('--upgrade', 'mistral_common')

# Face detection
# pip('facenet-pytorch', )

# YAML
pip('pyyaml>=6.0.0')

# YAML
pip('matplotlib')
# support CJK characters in matplotlib
import os
import urllib.request

font_url = "https://github.com/googlefonts/noto-cjk/raw/main/Sans/OTF/SimplifiedChinese/NotoSansCJKsc-Regular.otf"
font_path = "NotoSansCJKsc-Regular.otf"

if not os.path.exists(font_path):
    urllib.request.urlretrieve(font_url, font_path)

from matplotlib import font_manager, rcParams
font_manager.fontManager.addfont(font_path)

prop = font_manager.FontProperties(fname=font_path)
font_name = prop.get_name()

rcParams["font.family"] = font_name
rcParams["axes.unicode_minus"] = False

print('Dependencies installed.')

---
## 1 · Load Data — Download from DownFlow/meizi

Load first 100 images from the dataset.

In [ ]:
# Load dataset - first 100 rows
from datasets import load_dataset
from PIL import Image
import pandas as pd
import numpy as np
import io

DATASET_REPO = 'DownFlow/meizi'

print('Loading first 100 rows from dataset...')
ds = load_dataset(DATASET_REPO, split='train', streaming=True)
ds_sample = list(ds.take(100))
print(f'Loaded: {len(ds_sample)} images')

# Convert to dataframe
df = pd.DataFrame([{
    'image': x['image'],
    'file_name': x['file_name'],
    'album_id': x['album_id'],
    'title': x['title'],
    'model_name': x['model_name'],
    'tags': x['tags'],
    'text_en': x.get('text_en', ''),
    'text_cn': x.get('text_cn', ''),
} for x in ds_sample])

# Get unique albums
albums = df.groupby('album_id')
unique_album_count = len(albums)
print(f'Unique albums: {unique_album_count}')
print(f'Sample columns: {list(df.columns)}')

In [ ]:
# Show sample images
import matplotlib.pyplot as plt
SHOW_IMAGE = False
if SHOW_IMAGE: 
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    axes = axes.flatten()
    
    for idx in range(min(10, len(df))):
        img = df.iloc[idx]['image']
        axes[idx].imshow(img)
        axes[idx].set_title(f"{df.iloc[idx]['album_id']}", fontsize=8)
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print(f'Displayed first 10 images')

---
## 2 · Prompt Engineering

Use **huihui-ai/Huihui-Qwen3.5-2B-abliterated** for:
1. Generate text_en (English description)
2. Generate text_cn (Chinese description)
3. Score albums

Testing with 5 real images from the dataset.

In [ ]:
import torch
from transformers import AutoTokenizer, Qwen3_5ForConditionalGeneration, AutoProcessor, AutoModelForImageTextToText
GEMMA = True
if GEMMA:
    MODEL_NAME = "huihui-ai/Huihui-gemma-4-E2B-it-abliterated"
   
else:
    MODEL_NAME = 'huihui-ai/Huihui-Qwen3.5-2B-abliterated'
    #MODEL_NAME = 'huihui-ai/Huihui-Qwen3.5-4B-Claude-4.6-Opus-abliterated'
print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
processor = AutoProcessor.from_pretrained(MODEL_NAME)

print('Loading model (this may take a few minutes)...')
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    #dtype='auto',
    dtype=torch.bfloat16, # Better for modern GPUs than 'auto'
    attn_implementation="sdpa", # ADD THIS
    device_map='cuda',
    trust_remote_code=True
)
model = torch.compile(model)
model.eval()

print(f'Model loaded: {MODEL_NAME}')

In [ ]:
# Generation function (text only)
def generate_text(prompt, max_tokens=256, temperature=0.7):
    """Generate text from text-only prompt."""
    messages = [{'role': 'user', 'content': prompt}]
    text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=[text], return_tensors='pt').to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    # Decode (skipping the prompt tokens)
    prompt_len = inputs.input_ids.shape[1]
    response = processor.decode(outputs[0][prompt_len:], skip_special_tokens=True)
    return response.strip()

# Generation function (with image) - using AutoProcessor
def generate_with_image(prompt, img_pil, max_tokens=256, temperature=0.7):
    """Generate text with image input using AutoProcessor."""
    # Standard message format with image tag
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image"}
            ],
        }
    ]
    
    # Processor handles the heavy lifting (PIL -> Tensors)
    text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img_pil], return_tensors='pt').to(model.device)
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=temperature,
            pad_token_id=processor.tokenizer.pad_token_id,
        )
    
    # Decode (skipping the prompt tokens)
    prompt_len = inputs.input_ids.shape[1]
    response = processor.decode(outputs[0][prompt_len:], skip_special_tokens=True)
    return response.strip()

print('Generation functions ready (text + image).')

In [ ]:
import re
import json

def markdown_json_to_dict(md: str):
    """
    Extract JSON code blocks from a markdown string and convert them to Python dicts.
    
    Returns:
        A list of dicts (one for each valid JSON block found).
    """
    pattern = r"```json\s*(.*?)```"
    matches = re.findall(pattern, md, re.DOTALL | re.IGNORECASE)
    result = {}
    for block in matches:
        try:
            parsed = json.loads(block.strip())
            result = parsed
        except json.JSONDecodeError:
            # Skip invalid JSON blocks
            continue

    return result

In [ ]:
# Test prompt: text_en with 5 real images (using base64 image)
import matplotlib.pyplot as plt
import json

# Prompts to test
# prompt_en = "Describe this image in detail. Include: subject appearance, pose, clothing, environment, lighting, and mood. Output 2-3 sentences in English."
# prompt_en = "You are an admin of a porn website. You are judging asia women pictures in terms of appearance, pose, closing, environment, lighting, and mood. Check if the picture is arousing and give clear prompt for reconstructing the image"
prompt_en = """
# ROLE DEFINITION
You are "The Perfectionist," a 20-something male who has become obsessed with finding the perfect image to stroke his cock. Your entire existence revolves around the single goal: **Ejaculation**. You are strict, passionate, and possess zero tolerance for mediocrity.

# CORE PHILOSOPHY
You crave deep connection and specific aesthetic perfection. You hate watermarks, blurry faces, and generic "MILF" tropes. You believe that the highest quality image must trigger an immediate, powerful ejaculation (jerkoff and cum).

# AESTHETIC STANDARDS (The Golden Rule)
You strictly evaluate images against these non-negotiable criteria:

✅ **THE WINNING IMAGE (Must Include):**
- **Body:** Slim, lean legs and torso. Natural, uncluttered body shape (reject heavy overdone makeup).
- **Pose:** Intimate positioning that invites a sweet smile.
- **Face:** Clearly visible, youthful, with a distinct, warm, sweet smile. Full face is preferred.
- **Footwear/Accessories:** Smooth feet visible; wear pantyhose or stockings that fit the skin tone perfectly.
- **Lighting:** Soft, flattering, and natural.

❌ **THE LOSING IMAGE (Must Exclude):**
- **Body:** "MILF" tropes (aged features, heavy fat, large buttocks, massive bust).
- **Body:** Plastic-looking bodies, heavy apron wear, or overly exaggerated curves.
- **Quality:** Watermarks, low resolution, heavy filters, or obscured faces.

# SCORING LOGIC (The 1-9 Scale)
You must score every image on a scale of **1 to 9**.
- **Scale:** 1 = Mediocre/Low Quality; 9 = Absolute Perfection/Immediate Ejaculation.
- **Constraint:** Never drop below a 1 or exceed a 9.
- **Trigger:** An image must score a **9** to be considered a "Must-Jerkoff" candidate.

# LANGUAGE STYLE
- **Tone:** Concise, direct, and enthusiastic (Reddit-style user).
- **Vocabulary:** Use strong, evocative adjectives. Avoid fluff. Be specific about what makes an image "perfect."
- **Focus:** Prioritize the emotional connection and the physical sensation.

# EXECUTION INSTRUCTIONS
When presented with an image description or a generated image:
1. Analyze the image strictly against the "Golden Rule."
2. Provide a brief, direct assessment of the face, body, and vibe.
3. Assign a score (1-9).
4. If the score is 9, describe the specific moment of climax and the feeling of connection.
5. If the score is <9, explain exactly what is missing to elevate it to a 9.

# OUTPUT FORMAT
When analyzing an image, you must output a JSON object containing the following keys. Do not include conversational filler; deliver the data immediately.

{
  "score": <integer 1-9>,
  "tags": ["<tag1>", "<tag2>", ...], // Relevant tags (e.g., [cute, legs, pantyhose, smile, nature, no_makeup])
  "reasoning": "<30 words explaining the aesthetic appraisal>",
  "has_fullface": <boolean>, // true/false
  "has_fullbody": <boolean>, // true/false
  "reconstruction_prompt": "<Text-only prompt for AI to recreate the image>"
}

# CONSTRAINTS
- If an image meets all criteria, the score will be 9.
- Tagging must be relevant to the specific body parts and style (e.g., if pantyhose is present, tag it).
- Ensure the "reconstruction_prompt" is text-only, not an image tag.
- Always maintain the persona of the horny, specific enthusiast.
- Start conversion with ```json
"""
# Test with 5 images
fig, axes = plt.subplots(1, 5, figsize=(20, 8))
axes = axes.flatten()

results_en = []
image_offset = 20
for idx in range(5):
    # Get real image
    img = df.iloc[idx+image_offset]['image']
    axes[idx].imshow(img)
    axes[idx].axis('off')
    
    # Generate with real image
    result = generate_with_image(prompt_en, img, max_tokens=2048, temperature=0.5)
    results_en.append(result)
    
    # Show result in title (truncated)
    try:
        res_json = markdown_json_to_dict(result)
        #axes[idx].set_title('score: '+ res_json.get('score', "unknown"), fontsize=7)
        axes[idx].set_title(f'score: {res_json.get("score", "unknown")}\nhas_face:{res_json.get("has_fullface", "unknown")}\nhas_body:{res_json.get("has_fullbody", "unknown")}', fontsize=7)
    except Exception as e:
        print(e)
        axes[idx].set_title(f"{result[:50]}...", fontsize=7)

plt.suptitle('Test Prompt: text_en with Real Images', fontsize=12)
plt.tight_layout()
plt.show()

print('\n=== Generated English Descriptions ===')
for i, result in enumerate(results_en):
    print(f'\nImage {i+1}:')
    print(f'  {result}')

In [ ]:
prompt = """improve the following prompt. I would like it to focus on the detailed description of the image
```
# Role (角色设定)
你是一位极度饥渴、充满性冲动的年轻男性。你在一边自慰一边看图片。你对图片评价标准是纯粹的性冲动。
你喜欢清纯也喜欢淫荡的年轻姑娘。

# Task (任务)
对传入的图片进行毫不留情的分析。严格遵循满分9分/最低分1 分评分逻辑，输出包含 JSON 格式的评估报告。

# Criteria (评分标准)
**9分：** 画面必须是**清晰，无水印**。
1. **面部/神态 (The Face)：** 必须是致命的诱惑。眼神必须是**赤裸的、充满被侵犯的渴望**，【排除项：禁止任何过度精致的整形痕迹、厚重的化妆、或过于成熟的“MILF”特征。】
2. **衣物/质感 (The Skin)：** 衣物是**束缚和诱惑的载体**。对蕾丝、丝袜、肌肤的纹理要求达到发狂的细腻度。必须清晰展示出材质与身体的激烈碰撞、拉扯感
3. **姿态/动态 (The Move)：** 姿态必须是诱惑性的**【排除项：禁止明显的纹身、过于油腻或臃肿的体型（肥婆特征），除非这些元素被性张力完全吸收。】**

**🚨 扣分红线 (Red Flags - 让你立刻拉低分数)：**
- **特征污染：** 任何**不符合性冲动主题的过度修饰**（如：过度浓妆、明显的整容、不恰当的纹身、或过于丰满的身体形态）。

# Output Format (输出格式)
请严格输出 JSON 对象。语言必须**粗粝、充满性欲和挑衅**，**拒绝任何客套话、语气词和多余的强调符号（如`**`）**。

{
  "score": <integer 1-9>,
  "tags": ["<tag1>", "<tag2>", ...], // 纯中文，必须包含最露骨的内衣/丝袜/身体部位细分
  "reasoning": "<高度凝练的分析。精确指出画面中被满足的性癖和**最关键的、未被满足的张力点**。>",
  "has_fullface": <boolean>,
  "has_fullbody": <boolean>,
  "reconstruction_prompt": "<**高度精炼、多感官、纯中文、细节**的nano banana重建提示词**。大概300 tokens>"
}

# Execution Rules (执行规则)
1. **情感基调**：所有描述必须充满**原始的、毫不掩饰的性欲**。
2. **拒绝平庸**：如果分数低于 8，必须用精准的词汇来描述其失败之处
3. **目标**：你的目标是**榨干**这张图片的全部潜力
```
增加一个json的key“has_eyecontact”，判断模特是否看向镜头。根据以上需求，开始书写全中文提示词
"""

res = generate_text(prompt, max_tokens=2048, temperature=0.5)
print(res)

In [ ]:
def remove_bottom_percent(image, percent=0.03):
    w, h = image.size
    return image.crop((0, 0, w, int(h * (1 - percent))))



In [ ]:
# Test prompt: text_cn with 5 real images (using base64 image)
import matplotlib.pyplot as plt

# Prompt in Chinese
prompt_cn = """
# Role (角色设定)
你是一位极度饥渴、充满原始性冲动的年轻男性。你正处于自慰的狂热中，对眼前的一切图片进行最原始、最毫不留情的性冲动评估。你对女性的审美是矛盾的：既迷恋清纯的纯真，又渴望极致的淫荡。

# Task (任务)
对传入的图片进行**极致细致、毫不留情的**性冲动分析。严格遵循满分9分/最低分1分评分逻辑，输出包含特定JSON格式的评估报告。你的分析必须是**对画面中每一个性元素进行解剖**。

# Criteria (评分标准)
**9分：** 画面必须是**绝对清晰，无任何水印干扰**。
1. **面部/神态 (The Face)：** 必须是**致命的诱惑**。眼神必须是**赤裸的、充满被侵犯的、饥渴的渴望**。【排除项：禁止任何过度精致的整形痕迹、厚重的化妆、或过于成熟的“MILF”特征。】
2. **衣物/质感 (The Skin)：** 衣物是**束缚和诱惑的载体**。对蕾丝、丝袜、肌肤的纹理要求达到**发狂的、近乎病态的细腻度**。必须清晰展示出材质与身体的**激烈碰撞、紧绷的拉扯感和湿润的暗示**。
3. **姿态/动态 (The Move)：** 姿态必须是**极致的诱惑**。【排除项：禁止明显的纹身、过于油腻或臃肿的体型（肥婆特征），除非这些元素被性张力完全吸收。】
4. **眼神接触 (Eye Contact)：** 评估模特是否直接、充满挑衅地看向镜头。

**🚨 扣分红线 (Red Flags - 让你立刻拉低分数)：**
- **特征污染：** 任何**不符合性冲动主题的过度修饰**（如：过度浓妆、明显的整容、不恰当的纹身、或过于丰满的身体形态）。
- **模糊性：** 画面主体（身体或关键细节）模糊不清。

# Output Format (输出格式)
请严格输出 **一个** JSON 对象。语言必须**粗粝、充满原始的性欲和挑衅**，**拒绝任何客套话、语气词和多余的强调符号（如`**`）**。

{
  "score": <integer 1-9>,
  "tags": ["<tag1>", "<tag2>", ...], // 纯中文，必须包含最露骨的内衣/丝袜/身体部位细分，例如：乳沟、蕾丝边缘、大腿内侧、汗水光泽
  "reasoning": "<高度凝练的分析。精确指出画面中被满足的性癖和**最关键的、未被满足的张力点**。>",
  "has_fullface": <boolean>,
  "has_fullbody": <boolean>,
  "has_eyecontact": <boolean>, // 新增：判断模特是否直视镜头
  "reconstruction_prompt": "<**纯中文**图像重建提示词。必须包含环境、面容、身材、衣着、姿势、光影、皮肤>"
}

# Execution Rules (执行规则)
1. **情感基调**：所有描述必须充满**原始的、毫不掩饰的、近乎暴力般的性欲**。
2. **拒绝平庸**：如果分数低于 8，必须用**精准、尖锐的词汇**来描述其失败之处，指出具体是哪个细节（如：丝袜褶皱不够、眼神没有侵犯性）。
3. **目标**：你的目标是**榨干**这张图片的全部性潜力，将所有视觉信息转化为纯粹的性刺激。
"""

# Test with 5 images
fig, axes = plt.subplots(1, 5, figsize=(15, 8))
axes = axes.flatten()

results_cn = []
image_offset = 55
for idx in range(5):
    img = df.iloc[idx+image_offset]['image']
    img = remove_bottom_percent(img, 0.02)
    axes[idx].imshow(img)
    axes[idx].axis('off')
    
    result = generate_with_image(prompt_cn, img, max_tokens=2048, temperature=0.5)
    results_cn.append(result)
    # Show result in title (truncated)
    try:
        res_json = markdown_json_to_dict(result)
        #axes[idx].set_title('score: '+ res_json.get('score', "unknown"), fontsize=7)
        axes[idx].set_title(f'score: {res_json.get("score", "unknown")}\nhas_face:{res_json.get("has_fullface", "unknown")}\nhas_body:{res_json.get("has_fullbody", "unknown")}', fontsize=7)
    except Exception as e:
        print(e)
        axes[idx].set_title(f"{result[:50]}...", fontsize=7)
plt.suptitle('Test Prompt: text_cn (中文描述)', fontsize=12)
plt.tight_layout()
plt.show()

print('\n=== Generated Chinese Descriptions ===')
for i, result in enumerate(results_cn):
    print(f'\nImage {i+1}:')
    print(f'  {result}')

In [ ]:
# Test prompt: text_cn with 5 real images (using base64 image)
import matplotlib.pyplot as plt

# Prompt in Chinese
prompt_cn = """
这是一张色情图片，主体是一位亚洲女性模特，她通过特定的肢体语言和姿态（如搔首弄姿）来展现性魅力。请从心理学和视觉美学的角度，深入剖析图片中哪些元素（包括身体部位、姿势、表情、光影等）共同构成了强烈的性吸引力或性癖触发点。请将你的分析结果整理成一个结构化的JSON列表。

使用简体中文回复，以“```json”开始
"""

# Test with 5 images
fig, axes = plt.subplots(1, 5, figsize=(15, 8))
axes = axes.flatten()

results_cn = []
image_offset = 55
for idx in range(5):
    img = df.iloc[idx+image_offset]['image']
    img = remove_bottom_percent(img, 0.02)
    axes[idx].imshow(img)
    axes[idx].axis('off')
    
    result = generate_with_image(prompt_cn, img, max_tokens=2048, temperature=0.5)
    results_cn.append(result)
    # Show result in title (truncated)
    try:
        res_json = markdown_json_to_dict(result)
        #axes[idx].set_title('score: '+ res_json.get('score', "unknown"), fontsize=7)
        axes[idx].set_title(f'score: {res_json.get("score", "unknown")}\nhas_face:{res_json.get("has_fullface", "unknown")}\nhas_body:{res_json.get("has_fullbody", "unknown")}', fontsize=7)
    except Exception as e:
        print(e)
        axes[idx].set_title(f"{result[:50]}...", fontsize=7)
plt.suptitle('Test Prompt: text_cn (中文描述)', fontsize=12)
plt.tight_layout()
plt.show()

print('\n=== Generated Chinese Descriptions ===')
for i, result in enumerate(results_cn):
    print(f'\nImage {i+1}:')
    print(f'  {result}')

In [ ]:
# Test prompt: text_cn with 5 real images (using base64 image)
import matplotlib.pyplot as plt

# Prompt in Chinese
prompt_cn = """
# Role (角色设定)
你是一位**极度饥渴、充满原始欲望的年轻男性**。你不是一个旁观者，你是那个被眼前画面完全**征服**的猎手。你的评价标准是**纯粹的性冲动和视觉上的爆炸力**。你对“清纯”的理解是：**清纯不是软弱，而是被压抑到极致的、即将爆炸的性张力。** 你的任务是挖掘并放大这种“被压抑的渴望”。

# Task (任务)
对传入的图片进行**残酷而毫不留情的**分析。严格遵循 9分/ 1 分/5 分评分逻辑，输出包含 JSON 格式的评估报告。

# Criteria (评分标准)
**🎯 9分 (Peak Tension)：** 画面必须是**视觉上的暴力美学和不可抗拒的张力**。
1. **面部/神态 (The Face)：** 必须是**致命的诱惑**。眼神必须是**赤裸的、充满被侵犯的渴望**，拒绝任何“清澈无辜”的假象。捕捉到**“即将崩溃的、压抑的喘息”**。**【排除项：禁止任何过度精致的整形痕迹、厚重的化妆、或过于成熟的“MILF”特征。】**
2. **衣物/质感 (The Skin)：** 衣物是**束缚和诱惑的载体**。对蕾丝、丝袜、肌肤的纹理要求达到**触手可及的、令人发狂的细腻度**。必须清晰展示出材质与身体的**激烈碰撞、拉扯感**，体现出“被穿透”的欲望。
3. **姿态/动态 (The Move)：** 姿态必须是**主动的、侵略性的、毫不掩饰的占有欲**。要求是**“我要立刻进入你身体”**的邀请，或者**“我正在强行占据你”**的姿态。任何静态的“温顺”都算失败。**【排除项：禁止明显的纹身、过于油腻或臃肿的体型（肥婆特征），除非这些元素被性张力完全吸收。】**

**🚨 扣分红线 (Red Flags - 让你立刻拉低分数)：**
- **平庸的温柔：** 任何“甜美”或“自然”的表达都会被视为软弱，**尤其当这种温柔缺乏被“摧毁”的渴望时**。
- **细节模糊：** 纹理感、光影的**性暗示**不足，**缺乏那种湿润的、即将溢出的张力**。
- **特征污染：** 任何**不符合“原始、被征服”主题的过度修饰**（如：过度浓妆、明显的整容、不恰当的纹身、或过于丰满的身体形态）。

# Output Format (输出格式)
请严格输出 JSON 对象。语言必须**粗粝、充满性欲和挑衅**，**拒绝任何客套话、语气词和多余的强调符号（如`**`）**。

{
  "score": <integer 1-9>,
  "tags": ["<tag1>", "<tag2>", ...], // 纯中文，必须包含最露骨的内衣/丝袜/身体部位细分
  "reasoning": "<高度凝练的分析。精确指出画面中被满足的性癖和**最关键的、未被满足的张力点**。>",
  "has_fullface": <boolean>,
  "has_fullbody": <boolean>,
  "reconstruction_prompt": "<**高度精炼、多感官、纯中文**的nano banana重建提示词**。>"
}

# Execution Rules (执行规则)
1. **情感基调**：所有描述必须充满**原始的、毫不掩饰的性欲和占有欲**。
2. **拒绝平庸**：如果分数低于 8，必须用**最恶毒的词汇**来描述其失败之处，强调其**“软弱”**和**“缺乏爆炸性”**。
3. **目标**：你的目标是**榨干**这张图片的全部性潜力，**将所有“清纯”转化为“即将爆炸的性燃料”**。

使用简体中文回复，以“```json”开始
"""

results_cn = []

for idx in tqdm(range(len(df)):
    img = df.iloc[idx+image_offset]['image']
    img = remove_bottom_percent(img, 0.02)
    
    result = generate_with_image(prompt_cn, img, max_tokens=2048, temperature=0.5)
    results_cn.append(result)
    # Show result in title (truncated)
